# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit2003-dev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract formalizes the grain, time horizons, field taxonomies, and integrity constraints for the search intelligence refresh dataset before modeling begins.

## 1. Unit of analysis + time window

**Unit of Analysis:**
- **One row = one unique content item (URL/article)**, uniquely identified by the pseudonymous hash `content_id`.
- The dataset represents a portfolio cross-section across **32 distinct enterprise clients** (`client_id`).
- Each row aggregates performance and metadata attributes specifically for that single content asset.

**Time Window:**
- **Overall observation window:** A trailing **90-day observation window** terminating at snapshot export time (all metrics suffixed with `_90d`).
- **Comparative sub-windows (momentum components):**
  - `last_30d`: Days 1–30 trailing back from export.
  - `prev_30d`: Days 31–60 trailing back from export.
  - The relative delta `(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d` defines `trend_pct` and categorizes `trend_direction`.

In [ ]:
import os, sys, subprocess
import pandas as pd
import numpy as np
import duckdb

# Path resolution: works locally or in Colab
data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    for candidate in ["../../data/raw/content_refresh_anonymized.csv", "../data/raw/content_refresh_anonymized.csv"]:
        if os.path.exists(candidate):
            data_path = candidate
            break

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

# 1. Grain Verification: content_id must be strictly unique
con = duckdb.connect()
con.register("df_raw", df)

grain_check = con.execute("""
    SELECT content_id, COUNT(*) AS cnt
    FROM df_raw
    GROUP BY content_id
    HAVING COUNT(*) > 1
""").df()

print(f"Grain violations (duplicate content_id): {len(grain_check)}")
print(f"Distinct content items: {df['content_id'].nunique():,}")
print(f"Distinct client portfolios: {df['client_id'].nunique():,}")

# 2. Window Summary Metrics
window_summary = con.execute("""
    SELECT 
        MIN(days_with_impressions) AS min_days_imp,
        MAX(days_with_impressions) AS max_days_imp,
        MIN(content_age_days) AS min_content_age,
        MAX(content_age_days) AS max_content_age,
        MIN(impressions_90d) AS min_imp_90d,
        AVG(impressions_90d) AS avg_imp_90d,
        AVG(impressions_last_30d) AS avg_imp_last30,
        AVG(impressions_prev_30d) AS avg_imp_prev30
    FROM df_raw
""").df()
display(window_summary)


## 2. Fields: feature / label / context / excluded

Every single field in the dataset is classified into exactly one of four operational buckets:

1. **Feature (24 columns):** Observational inputs knowable prior to prediction time.
   - *Search Volume & Keyword context:* `search_volume`, `competition`, `cpc`.
   - *Content properties:* `word_count`, `char_count`, `content_age_days`, `days_since_last_update`.
   - *Search & Analytics aggregates (90d):* `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`.
   - *Derived efficiency rates:* `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
   - *Categorical context:* `content_type`, `main_intent`.

2. **Label / Proxy Target (2 columns):**
   - `is_declining_label`: Primary binary target (`1` if `trend_direction == 'down'`, else `0`).
   - `trend_pct`: Continuous relative trajectory; source of the label.

3. **Context / Grouping (4 columns):**
   - `content_id`: Unique row identifier (joins/traceability only, never a feature).
   - `client_id`: Brand grouping key; strictly reserved for client-holdout validation splits to avoid cross-client data leakage.
   - `provider_used`, `model_used`: Generative metadata, excluded from model training to prevent spurious LLM branding bias.

4. **Excluded (14 columns):**
   - `trend_direction`, `trend_pct`: Direct label components (target leakage).
   - `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`: Direct constituents of the label calculation; using them leaks the target period directly into features.
   - `competition_level`, `age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`: Arbitrary coarse discretized bins of continuous metrics; continuous representations preserve full granularity and eliminate arbitrary binning boundaries.

In [ ]:
field_taxonomy = {
    "features": [
        "search_volume", "competition", "cpc", "word_count", "char_count",
        "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
        "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
        "days_with_impressions", "days_with_sessions", "content_age_days",
        "days_since_last_update", "ctr", "avg_position", "engagement_rate",
        "scroll_rate", "ai_traffic_pct", "content_type", "main_intent"
    ],
    "labels": ["trend_direction", "trend_pct"],
    "context": ["content_id", "client_id", "provider_used", "model_used"],
    "excluded": [
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
        "competition_level", "age_tier", "freshness_tier", "word_count_tier",
        "char_count_tier", "impression_tier", "position_tier", "age_tier_order"
    ]
}

total_categorized = sum(len(v) for v in field_taxonomy.values())
print(f"Total columns in dataset: {df.shape[1]}")
print(f"Total columns classified: {total_categorized}")
assert total_categorized == df.shape[1], "Mismatch in field categorization!"

for category, cols in field_taxonomy.items():
    print(f"  - {category.capitalize()} ({len(cols)} columns): {', '.join(cols[:5])}...")


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify the dataset integrity via DuckDB queries:
1. Confirm zero duplicate `content_id` rows.
2. Analyze systematic missingness patterns across `content_type`.
3. Audit rate calculations, nulls, and sentinel zero values (`avg_position = 0` meaning unranked/no-data).

In [ ]:
# 1. Systematic Missingness Analysis across Content Types
missing_analysis = con.execute("""
    SELECT 
        content_type,
        COUNT(*) AS total_pages,
        ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS missing_search_vol_pct,
        ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS missing_word_count_pct,
        ROUND(AVG(CASE WHEN cpc IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS missing_cpc_pct,
        ROUND(AVG(CASE WHEN avg_position = 0 THEN 1.0 ELSE 0.0 END) * 100, 2) AS zero_position_pct
    FROM df_raw
    GROUP BY content_type
    ORDER BY total_pages DESC
""").df()

print("Missingness by Content Type:")
display(missing_analysis)

# 2. Rate and Boundary Checks
rate_checks = con.execute("""
    SELECT 
        'ctr' AS metric, MIN(ctr) AS min_val, AVG(ctr) AS avg_val, MAX(ctr) AS max_val, COUNT(CASE WHEN ctr > 100 THEN 1 END) AS count_gt_100
    UNION ALL
    SELECT 
        'scroll_rate', MIN(scroll_rate), AVG(scroll_rate), MAX(scroll_rate), COUNT(CASE WHEN scroll_rate > 100 THEN 1 END)
    UNION ALL
    SELECT 
        'ai_traffic_pct', MIN(ai_traffic_pct), AVG(ai_traffic_pct), MAX(ai_traffic_pct), COUNT(CASE WHEN ai_traffic_pct > 100 THEN 1 END)
""").df()

print("\nRate Metrics Validation (note: scroll_rate and ai_traffic_pct can validly exceed 100 due to independent log counters):")
display(rate_checks)

# 3. Target Distribution Check
target_check = con.execute("""
    SELECT 
        trend_direction,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_total
    FROM df_raw
    GROUP BY trend_direction
    ORDER BY row_count DESC
""").df()

print("\nLabel / Target Distribution:")
display(target_check)


## 4. Data limits

**What this data can and cannot answer:**
1. **Observational vs. Causal:** This dataset is a cross-sectional observational snapshot. It reveals statistical associations (e.g., stale pages with high impressions correlate with negative momentum), but cannot prove that refreshing a page causally recovers its ranking without an A/B or staggered rollout design.
2. **Proxy Target Horizon:** `trend_direction == 'down'` compares two adjacent 30-day historical slices (`last_30d` vs `prev_30d`) within the same export snapshot rather than tracking true forward post-intervention performance.
3. **Systematic Null Patterns:** Notice above that `feedly article` pages have exactly 100% missing keyword data and 28.5% missing word counts. Imputing nulls with blind zeros would conflate content type with feature magnitude. Indicator flags (`has_search_volume`, `has_word_count`) are required.
4. **Panel Asymmetry:** Client portfolio sizes range from under 200 pages to several thousand pages. Validation must stratify or group by `client_id` to prevent high-volume clients from biasing decision thresholds.

In [ ]:
# Client Concentration Audit
client_dist = con.execute("""
    SELECT 
        COUNT(DISTINCT client_id) AS total_clients,
        MIN(client_page_count) AS min_pages_per_client,
        MEDIAN(client_page_count) AS median_pages_per_client,
        MAX(client_page_count) AS max_pages_per_client,
        ROUND(AVG(decline_rate), 3) AS avg_client_decline_rate
    FROM (
        SELECT 
            client_id,
            COUNT(*) AS client_page_count,
            AVG(CASE WHEN trend_direction = 'down' THEN 1.0 ELSE 0.0 END) AS decline_rate
        FROM df_raw
        GROUP BY client_id
    )
""").df()

print("Client Concentration and Diversity:")
display(client_dist)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.